# THESIS-007 — Experiment 2A: K8s Isolation (SQ2/SQ3)
**Story Points:** 5 | **Status:** IN PROGRESS (running)

Experiment 2A is currently running in the background (PID ~34392, log: `/tmp/exp2a.log`).
L1 complete (3/3 reps). Answers SQ2 (pod isolation) and SQ3 (scheduling overhead).

**To check progress:**
```bash
tail -f /tmp/exp2a.log
kubectl get pods -n dagster
```


## Acceptance Criteria
- [ ] 18 batches completed (6 levels x 3 reps)
- [ ] `k8s_pod_metrics.csv` per batch
- [ ] `pod_timing.csv` per batch with scheduling/startup timestamps
- [ ] `dagster_runs.csv` per batch
- [ ] All `metadata.json` files present
- [ ] Pod isolation confirmed (each run has its own pod)

## Live progress monitor

In [ ]:
import subprocess
r = subprocess.run(['tail', '-20', '/tmp/exp2a.log'], capture_output=True, text=True)
print('=== /tmp/exp2a.log (last 20 lines) ===')
print(r.stdout if r.returncode == 0 else 'Log not found')
print('\n=== kubectl get pods -n dagster ===')
r2 = subprocess.run(['kubectl', 'get', 'pods', '-n', 'dagster'], capture_output=True, text=True)
print(r2.stdout)

## Check data progress

In [ ]:
import os
from pathlib import Path
exp_dir = Path('../data/raw/exp2-kubernetes-isolation/part-a')
levels = [1, 2, 3, 5, 7, 10]
reps   = [1, 2, 3]
done_count = 0
for level in levels:
    for rep in reps:
        run_dir = exp_dir / f'L{level}' / f'run{rep}'
        done = run_dir.exists() and (run_dir / 'dagster_runs.csv').exists()
        status = 'DONE' if done else 'TODO'
        if done:
            done_count += 1
        print(f'  L{level} rep{rep}: {status}')
print(f'\nProgress: {done_count}/18 batches complete')

## Launch Exp2A (if not already running)

In [ ]:
import subprocess, os
log = '/tmp/exp2a.log'
r = subprocess.run(['pgrep', '-f', 'run_experiment.sh.*exp2a'], capture_output=True, text=True)
if r.returncode == 0:
    pids = r.stdout.strip()
    print(f'Experiment 2A is RUNNING (PIDs: {pids})')
    print(f'Log: {log}')
else:
    print('Experiment 2A is NOT running.')
    print('To start:')
    print('  cd .. && DAGSTER_PORT=3001 bash scripts/run_experiment.sh exp2a k8s > /tmp/exp2a.log 2>&1 &')
    print('  # or: make exp2a-k8s')